In [ ]:
pip uninstall keras -y



SyntaxError: invalid syntax (351194048.py, line 1)

In [2]:
pip install tensorflow==2.12

ERROR: Could not find a version that satisfies the requirement tensorflow==2.12 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.12


In [1]:
!pip install mediapipe
!pip install moviepy
!pip install tensorflow

In [2]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import vision, BaseOptions
import tensorflow as tf
from tensorflow.keras.applications.mobilenet import preprocess_input

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1. Cropping Video

Hand Detector Setup

In [6]:
MODEL_PATH = "/content/hand_landmarker.task"

base_options = BaseOptions(model_asset_path=MODEL_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2
)

detector = vision.HandLandmarker.create_from_options(options)

Load Video Frames Function

In [7]:
def load_frames_from_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return frames

Making Box Function

In [8]:
def get_static_hand_bbox(frames, padding=0.2):
    global_min_x, global_min_y = 1.0, 1.0
    global_max_x, global_max_y = 0.0, 0.0

    found = False
    h, w = frames[0].shape[:2]

    for frame in frames:
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
        result = detector.detect(mp_image)

        if result.hand_landmarks:
            found = True
            for hand in result.hand_landmarks:
                for lm in hand:
                    global_min_x = min(global_min_x, lm.x)
                    global_min_y = min(global_min_y, lm.y)
                    global_max_x = max(global_max_x, lm.x)
                    global_max_y = max(global_max_y, lm.y)

    if not found:
        # fallback: return full frame
        return 0, 0, w, h

    # convert to pixels
    box_w = global_max_x - global_min_x
    box_h = global_max_y - global_min_y

    x1 = max(0, int((global_min_x - padding * box_w) * w))
    y1 = max(0, int((global_min_y - padding * box_h) * h))
    x2 = min(w, int((global_max_x + padding * box_w) * w))
    y2 = min(h, int((global_max_y + padding * box_h) * h))

    return x1, y1, x2, y2

Crop Frames Function

In [9]:
def crop_frames_with_bbox(frames, bbox):
    """
    Crop all frames using the same bounding box.

    Args:
        frames (list of np.array): list of frames (H,W,C)
        bbox (tuple): (x1, y1, x2, y2)

    Returns:
        list of cropped frames
    """
    x1, y1, x2, y2 = bbox

    cropped_frames = []
    for f in frames:
        crop = f[y1:y2, x1:x2]
        cropped_frames.append(crop)

    return cropped_frames

2. Making Dynamic image from cropped video (It will differ from here for every method)

Dynamic Image Function

In [10]:
def get_dynamic_image(frames, normalized=True):
    """ Takes a list of frames and returns either a raw or normalized dynamic image."""
    num_channels = frames[0].shape[2]
    channel_frames = _get_channel_frames(frames, num_channels)
    channel_dynamic_images = [_compute_dynamic_image(channel) for channel in channel_frames]

    dynamic_image = cv2.merge(tuple(channel_dynamic_images))
    if normalized:
        dynamic_image = cv2.normalize(dynamic_image, None, 0, 255, norm_type=cv2.NORM_MINMAX)
        dynamic_image = dynamic_image.astype('uint8')

    return dynamic_image


def _get_channel_frames(iter_frames, num_channels):
    """ Takes a list of frames and returns a list of frame lists split by channel. """
    frames = [[] for channel in range(num_channels)]

    for frame in iter_frames:
        for channel_frames, channel in zip(frames, cv2.split(frame)):
            channel_frames.append(channel.reshape((*channel.shape[0:2], 1)))
    for i in range(len(frames)):
        frames[i] = np.array(frames[i])
    return frames

def _compute_dynamic_image(frames):
    """ Adapted from https://github.com/hbilen/dynamic-image-nets """
    num_frames, h, w, depth = frames.shape

    # Compute the coefficients for the frames.
    coefficients = np.zeros(num_frames)
    for n in range(num_frames):
        cumulative_indices = np.array(range(n, num_frames)) + 1
        coefficients[n] = np.sum(((2*cumulative_indices) - num_frames) / cumulative_indices)

    # Multiply by the frames by the coefficients and sum the result.
    x1 = np.expand_dims(frames, axis=0)
    x2 = np.reshape(coefficients, (num_frames, 1, 1, 1))
    result = x1 * x2
    return np.sum(result[0], axis=0).squeeze()

#Reading Frames from Video
def load_frames_opencv(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return frames

Transfrom Vid -> Dynamic Image

In [11]:
def video_to_dynamic_image(video_path, output_size=(224,224)):

    frames = load_frames_opencv(video_path)
    if len(frames) == 0:
        raise ValueError("No frames found in video")

    # 2. Get hand bounding box
    bbox = get_static_hand_bbox(frames)

    # 3. Crop frames
    cropped_frames = crop_frames_with_bbox(frames, bbox)

# ----------------------------------------------------------------------------------
    # The process will differ after this (for vid processing or static image)!!
# ----------------------------------------------------------------------------------

    # 4. Create dynamic image
    dyn_img = get_dynamic_image(cropped_frames)

    # 5. Resize
    dyn_img = cv2.resize(dyn_img, output_size)

    return dyn_img

Test Time and Memory Usage

In [14]:
import os
import time
import psutil
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications.mobilenet import preprocess_input

process = psutil.Process(os.getpid())


model = tf.keras.models.load_model(
    "/content/final_mobilenet_model_final, 63.38.keras",
    compile=False,
    safe_mode=False
)

# I use sign_455783 as testing
video_folder = "/content/drive/MyDrive/FINAL RESULT/Dynamic Image Result/Testing Memory and Time Usage/test/House/sign_455783.mp4"
# model_path = "/final_mobilenet_model_final, 63.38.keras"

TRUE_LABEL = "HOUSE"

class_names = ['BUY','CAR','Eat','FRIEND','GO','GO-OUT',
               'HOUSE','Have','KISS-FIST','LOOK','Must',
               'NOT','NOW','PEOPLE','TEND']

# model = tf.keras.models.load_model(model_path)

results = []

for video in os.listdir(video_folder):

    video_path = os.path.join(video_folder, video)

    #Start Time
    start_time = time.time()
    start_mem = process.memory_info().rss / (1024**2)

    #Change vid -> Dynamic Img
    dyn_img = video_to_dynamic_image(video_path)

    dyn_img = dyn_img.astype("float32")
    dyn_img = preprocess_input(dyn_img)
    dyn_img = np.expand_dims(dyn_img, axis=0)

    pred = model.predict(dyn_img, verbose=0)
    pred_class = np.argmax(pred)
    pred_label = class_names[pred_class]

    #Stop Time
    end_time = time.time()
    end_mem = process.memory_info().rss / (1024**2)

    #Get data for time and memory consumption
    results.append({
        "video": video,
        "true_label": TRUE_LABEL,
        "pred_label": pred_label,
        "correct": int(pred_label == TRUE_LABEL),
        "time_sec": end_time - start_time,
        "memory_MB": end_mem - start_mem
    })

#Get Result
df = pd.DataFrame(results)

df.to_csv("single_class_test.csv", index=False)

print(df)

print("\nAccuracy (this class only):", df["correct"].mean())
print("Avg Time:", df["time_sec"].mean())
print("Avg Memory:", df["memory_MB"].mean())

TypeError: Could not deserialize class 'Functional' because its parent module keras.src.engine.functional cannot be imported. Full object config: {'module': 'keras.src.engine.functional', 'class_name': 'Functional', 'config': {'name': 'model_1', 'trainable': True, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_input_shape': [None, 224, 224, 3], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_2'}, 'registered_name': None, 'name': 'input_2', 'inbound_nodes': []}, {'module': 'keras.layers', 'class_name': 'Conv2D', 'config': {'name': 'Conv1', 'trainable': False, 'dtype': 'float32', 'filters': 32, 'kernel_size': [3, 3], 'strides': [2, 2], 'padding': 'same', 'data_format': 'channels_last', 'dilation_rate': [1, 1], 'groups': 1, 'activation': 'linear', 'use_bias': False, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module':